In [1]:
# Imports
import pandas as pd
import boto3
import os

In [ ]:
hour_df = pd.read_parquet("s3://dsan6000-wikipedia/hourly_parquet/20260901_040000.parquet")
display(hour_df.head())

,datetime,$schema,meta,id,type,namespace,title,title_url,comment,timestamp,...,wiki,parsedcomment,log_id,log_type,log_action,log_params,log_action_comment,patrolled,file_ts,file_ts_str
0,2026-09-01 04:00:02,/mediawiki/recentchange/1.0.0,{'uri': 'https://en.wikipedia.org/wiki/Jake_O%...,2.063884e+09,edit,0,Jake O'Donnell,https://en.wikipedia.org/wiki/Jake_O%27Donnell,Fix dash,1788235202,...,enwiki,Fix dash,NaN,NaN,NaN,NaN,NaN,None,2026-09-01 04:00:00,20260901_040000
1,2026-09-01 04:00:02,/mediawiki/recentchange/1.0.0,{'uri': 'https://en.wikipedia.org/wiki/Beverid...,2.063884e+09,edit,0,Beveridge Award,https://en.wikipedia.org/wiki/Beveridge_Award,Add source,1788235202,...,enwiki,Add source,NaN,NaN,NaN,NaN,NaN,None,2026-09-01 04:00:00,20260901_040000
2,2026-09-01 04:00:01,/mediawiki/recentchange/1.0.0,{'uri': 'https://en.wikipedia.org/wiki/1920_Un...,2.063884e+09,edit,0,1920 United States presidential election in Wa...,https://en.wikipedia.org/wiki/1920_United_Stat...,/* Counties that flipped from Democratic to Re...,1788235201,...,enwiki,"<span class=""autocomment""><a href=""/wiki/1920_...",NaN,NaN,NaN,NaN,NaN,None,2026-09-01 04:00:00,20260901_040000
3,2026-09-01 04:00:02,/mediawiki/recentchange/1.0.0,{'uri': 'https://en.wikipedia.org/wiki/1987_Ae...,2.063884e+09,edit,0,1987 Aegean crisis,https://en.wikipedia.org/wiki/1987_Aegean_crisis,added a bit of context,1788235202,...,enwiki,added a bit of context,NaN,NaN,NaN,NaN,NaN,None,2026-09-01 04:00:00,20260901_040000
4,2026-09-01 04:00:03,/mediawiki/recentchange/1.0.0,{'uri': 'https://en.wikipedia.org/wiki/Marvin_...,2.063884e+09,edit,0,Marvin Schwäbe,https://en.wikipedia.org/wiki/Marvin_Schw%C3%A4be,NaN,1788235203,...,enwiki,NaN,NaN,NaN,NaN,NaN,NaN,None,2026-09-01 04:00:00,20260901_040000


In [12]:
# Initialize S3 client
s3_client = boto3.client('s3')

# Specify the bucket with input data
bucket = 'dsan6000-wikipedia'
prefix = 'hourly_parquet/'

# Verify connection
try:
    response = s3_client.list_objects_v2(Bucket = bucket, Prefix = prefix)
    print("Successfully connected to S3!\nObjects available:", [obj['Key'] for obj in response.get('Contents', [])])
except Exception as e:
    print("Error! Couldn't connect to S3. See error:", e)

Successfully connected to S3!
Objects available: ['hourly_parquet/', 'hourly_parquet/20260901_040000.parquet', 'hourly_parquet/20260901_050000.parquet', 'hourly_parquet/20260901_060000.parquet', 'hourly_parquet/20260901_070000.parquet', 'hourly_parquet/20260901_080000.parquet', 'hourly_parquet/20260901_090000.parquet', 'hourly_parquet/20260901_100000.parquet', 'hourly_parquet/20260901_110000.parquet', 'hourly_parquet/20260901_120000.parquet', 'hourly_parquet/20260901_130000.parquet', 'hourly_parquet/20260901_140000.parquet', 'hourly_parquet/20260901_150000.parquet', 'hourly_parquet/20260901_160000.parquet', 'hourly_parquet/20260901_170000.parquet', 'hourly_parquet/20260901_180000.parquet', 'hourly_parquet/20260901_190000.parquet', 'hourly_parquet/20260901_200000.parquet', 'hourly_parquet/20260901_210000.parquet', 'hourly_parquet/20260901_220000.parquet', 'hourly_parquet/20260901_230000.parquet', 'hourly_parquet/20260902_000000.parquet', 'hourly_parquet/20260902_010000.parquet', 'hourly

In [ ]:
def download_parquet_files(bucket, prefix, output_subfolder, output_filepath):
    """
    Download all parquet files under a given S3 bucket and prefix, and save these files in a locally in a given location

    Parameters
    ----------
    bucket:
                        <description>
    prefix:
                        <description>
    output_filename:    list
                        <description>

    Returns
    -------
    list of str
    Local file paths of saved, downloaded files from S3 bucket.

    """

    # Make directory to store 
    output_dir = os.path.join(output_filepath, output_subfolder)
    os.makedirs(output_dir, exist_ok = True)

    # Initialize S3 Client
    s3_client = boto3.client('s3')

    # Grab connection to bucket & prefix
    response = s3_client.list_objects_v2(Bucket = bucket, Prefix = prefix)

    # Initializing empty list of saved filepaths for downloaded objects
    downloaded_objs = []

    try:
        # Loop through each object in bucket + prefix
        for obj in response.('Contents', []):

            # Get key of object
            key = obj['Key']

            # If files in folder are not .parquet, then skip
            if not key.endswith('.parquet'):
                continue

            # Get filename of obj & S3 filepath of obj
            filename = os.path.basename(key)
            s3_filepath = f"s3://{bucket}/{key}"

            # Store S3 parquet file in a pandas dataframe
            df = pd.read_parquet(s3_filepath)

            # Initialize output_path
            output_path = os.path.join(output_filepath, new_filename)

            # Establish new name for file
            new_filename = f"{filename}_downloaded"

            # Save dataframe to local parquet file
            df.to_parquet(output_path)

            # Append downloaded_objs list
            downloaded_objs.append(output_path)

    except Exception as e:
        print(f"Files were not able to be downloaded from the given S3 bucket. An error has occurred: {e}")

    return downloaded_objs



In [ ]:
# create and fill in a .gitignore file to ensure that these .parquet files downloaded into the data subfolder are not pushed to GitHub when you push your progress on the assignment